In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aaryaupi/umls-dataset/umls_bert_held_out (1).json
/kaggle/input/datasets/aaryaupi/umls-dataset/episodic_memory (30).tsv
/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_train (2).json
/kaggle/input/datasets/aaryaupi/umls-dataset/nations_held_out (7).json
/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_test (2).json
/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_all (3).json
/kaggle/input/datasets/aaryaupi/umls-dataset/nations_val (7).json
/kaggle/input/datasets/aaryaupi/umls-dataset/val_hard_results1 (1).json
/kaggle/input/datasets/aaryaupi/bert-dataset/val_hard_results.json
/kaggle/input/datasets/aaryaupi/bert-dataset/bert_training_data.json
/kaggle/input/datasets/aaryaupi/bert-dataset/held_out_results.json
/kaggle/input/datasets/aaryaupi/bert-dataset/bert_reranker_held_out.json


In [2]:
!pip install pykeen -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 16.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.1 MB/s eta 0:00:00


In [3]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires

In [4]:
from pykeen.datasets import Nations
from pykeen.pipeline import pipeline
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict

In [5]:
from pykeen.datasets import UMLS
dataset = UMLS()

df_train = pd.DataFrame(
    dataset.training.triples,
    columns = ["head" , "relation","tail"]
)

df_test = pd.DataFrame(
    dataset.testing.triples,
    columns = ["head" , "relation","tail"]
)

df_all = pd.concat([df_train , df_test],ignore_index = True)

entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

print(len(df_train))
print(len(df_test))
print(len(df_all))
print(f"Entities:      {df_all['head'].nunique()}")
print(f"Relations:     {df_all['relation'].nunique()}")

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


5216
661
5877
Entities:      135
Relations:     46


In [6]:
def show_neighborhood(entity, graph):
    print(f"\n{'='*50}")
    print(f"Neighborhood of: {entity}")
    print(f"{'='*50}")
    neighbors = graph.get(entity, [])
    print(f"Direct edges: {len(neighbors)}")
    print(f"\n{'Relation':<30} {'Tail':<20}")
    print("-" * 50)
    for relation, tail in sorted(neighbors, key=lambda x: x[0]):
        print(f"{relation:<30} {tail:<20}")

In [7]:
def build_type_constraints(df):
    """
    For each relation:
      - what tail entities appear, and how often?
      - what head entities appear, and how often?
      - what (head, tail) pairs co-occur?
    
    This gives agents the UMLS ontology signal:
    "given relation R and head H, what tail types are expected?"
    """
    from collections import defaultdict
    
    # how often does each tail appear for each relation
    rel_to_tail_counts = defaultdict(lambda: defaultdict(int))
    rel_to_head_counts = defaultdict(lambda: defaultdict(int))
    rel_to_pair_counts = defaultdict(lambda: defaultdict(int))
    
    for _, row in df.iterrows():
        h, r, t = row["head"], row["relation"], row["tail"]
        rel_to_tail_counts[r][t] += 1
        rel_to_head_counts[r][h] += 1
        rel_to_pair_counts[r][(h, t)] += 1
    
    # for each (relation, head): ranked list of expected tails
    # this is the core type constraint lookup
    rel_head_to_ranked_tails = defaultdict(dict)
    for r, pair_counts in rel_to_pair_counts.items():
        # group by head
        head_to_tails = defaultdict(dict)
        for (h, t), count in pair_counts.items():
            head_to_tails[h][t] = count
        for h, tail_counts in head_to_tails.items():
            total = sum(tail_counts.values())
            rel_head_to_ranked_tails[(r, h)] = {
                t: round(c / total, 4)
                for t, c in sorted(
                    tail_counts.items(), key=lambda x: -x[1]
                )
            }
    
    # for each relation: overall tail distribution
    rel_to_tail_dist = {}
    for r, tail_counts in rel_to_tail_counts.items():
        total = sum(tail_counts.values())
        rel_to_tail_dist[r] = {
            t: round(c / total, 4)
            for t, c in sorted(
                tail_counts.items(), key=lambda x: -x[1]
            )
        }
    
    return {
        "rel_head_to_ranked_tails": dict(rel_head_to_ranked_tails),
        "rel_to_tail_dist":         rel_to_tail_dist,
        "rel_to_tail_counts":       dict(rel_to_tail_counts),
        "rel_to_head_counts":       dict(rel_to_head_counts),
    }


def get_type_constraint_signal(head, relation, true_tail,
                                predicted, constraints):
    """
    UMLS replacement for path-based only_tail_has.
    
    Returns:
      type_fit_true  - how well true_tail fits the type slot
      type_fit_pred  - how well predicted fits the type slot  
      expected_tails - top-5 expected tails for (relation, head)
      only_true_has  - relations true_tail appears in that predicted doesn't
      only_pred_has  - relations predicted appears in that true_tail doesn't
    """
    rh_tails  = constraints["rel_head_to_ranked_tails"]
    tail_dist = constraints["rel_to_tail_dist"]
    
    # type fit scores — how expected is each candidate as tail?
    specific  = rh_tails.get((relation, head), {})
    general   = tail_dist.get(relation, {})
    
    # specific (this exact head) first, fallback to general
    type_fit_true = specific.get(true_tail) or general.get(true_tail, 0.0)
    type_fit_pred = specific.get(predicted) or general.get(predicted, 0.0)
    
    # top expected tails for this (relation, head) pair
    expected_tails = list(specific.keys())[:5] or list(general.keys())[:5]
    
    # relational profile difference
    # what relations does true_tail appear in vs predicted?
    tail_counts = constraints["rel_to_tail_counts"]
    
    true_tail_rels = {
        r for r, tails in tail_counts.items()
        if true_tail in tails
    }
    pred_rels = {
        r for r, tails in tail_counts.items()
        if predicted in tails
    }
    
    only_true = true_tail_rels - pred_rels
    only_pred = pred_rels - true_tail_rels
    shared    = true_tail_rels & pred_rels
    
    return {
        "type_fit_true":  type_fit_true,
        "type_fit_pred":  type_fit_pred,
        "type_gap":       round(type_fit_true - type_fit_pred, 4),
        "expected_tails": expected_tails,
        "only_true_has":  sorted(only_true),   # ← same field name, new meaning
        "only_pred_has":  sorted(only_pred),
        "shared_rels":    sorted(shared),
    }

In [8]:
result = pipeline(
    dataset="UMLS",
    model="RotatE",
    model_kwargs=dict(embedding_dim=256),
    optimizer="adam",
    optimizer_kwargs=dict(lr=1e-3),
    training_kwargs=dict(
        num_epochs=500,
        batch_size=512,
    ),
    negative_sampler="basic",
    negative_sampler_kwargs=dict(num_negs_per_pos=64),
    evaluator_kwargs=dict(filtered=True),
    random_seed=42,
)


winner_model = result.model

Training epochs on cuda:0:   0%|          | 0/500 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/661 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.20s seconds


In [9]:
mrr = result.metric_results.get_metric("mean_reciprocal_rank")
print("MRR:", mrr)


MRR: 0.7977453470230103


In [10]:
# the triples are integer IDs — convert to strings first
def build_graph(triples_array, id_to_entity, id_to_relation):
    """
    Converts integer ID triples → string adjacency dict.
    
    triples_array: numpy array of shape (N, 3) 
                   columns: [head_id, relation_id, tail_id]
    """
    graph = defaultdict(list)
    
    for h_id, r_id, t_id in triples_array:
        head     = id_to_entity[h_id]
        relation = id_to_relation[r_id]
        tail     = id_to_entity[t_id]
        graph[head].append((relation, tail))
    
    return graph

# rebuild id mappings cleanly
entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

# get raw integer arrays
train_triples = dataset.training.mapped_triples.numpy()
test_triples  = dataset.testing.mapped_triples.numpy()
all_triples   = np.vstack([train_triples, test_triples])

# build graph on training only
graph = build_graph(train_triples, id_to_entity, id_to_relation)

# also rebuild df_train and df_test correctly
df_train = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in train_triples
], columns=["head", "relation", "tail"])

df_test = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in test_triples
], columns=["head", "relation", "tail"])

df_all = pd.concat([df_train, df_test], ignore_index=True)

# verify
print(f"Train triples: {len(df_train)}")
print(f"Test triples:  {len(df_test)}")
print(f"Graph nodes:   {len(graph)}")
print(f"\nSample df_train:")
print(df_train.head(5).to_string(index=False))

Train triples: 5216
Test triples:  661
Graph nodes:   135

Sample df_train:
                head relation      tail
acquired_abnormality  affects      alga
acquired_abnormality  affects    animal
acquired_abnormality  affects  archaeon
acquired_abnormality  affects bacterium
acquired_abnormality  affects      bird


In [11]:
!pip install transformers

In [12]:
constraints = build_type_constraints(df_train)
print(f"Constraints built: {len(constraints['rel_to_tail_dist'])} relations")

Constraints built: 46 relations


In [13]:
import json, random, numpy as np, torch
from collections import defaultdict
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# ── config ────────────────────────────────────────────────
MAX_K        = 50
W_KGE        = 0.5
W_AGENT      = 0.3
W_OVERLAP    = 0.2
MAX_LENGTH   = 512
NEG_PRIORITY = ["model_topk", "structural", "similarity", "random"]

# ── load agent results ────────────────────────────────────
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/val_hard_results1 (1).json") as f:
    agent_results = json.load(f)

# ── load preprocessed data — BOTH splits ─────────────────
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_train (2).json") as f:
    train_data = json.load(f)

with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_test (2).json") as f:
    test_data = json.load(f)

# ── index covers both splits ──────────────────────────────
preprocessed_index = {
    (r["head"], r["relation"], r["tail"]): r
    for r in train_data + test_data
}

print(f"Agent results  : {len(agent_results)}")
print(f"Train records  : {len(train_data)}")
print(f"Test records   : {len(test_data)}")
print(f"Index size     : {len(preprocessed_index)}")

# agent confidence lookup from results
def load_agent_confidence(agent_results):
    lookup = {}
    for r in agent_results:
        agg    = r.get("aggregator", {})
        chosen = agg.get("chosen_agent", "B")
        agent_out = r.get(
            "agent_a" if chosen == "A" else "agent_b", {}
        )
        conf = float(agent_out.get("confidence", 0.5))

        # parse "(head, relation, tail)" string
        t = r.get("triple", "")
        if t.startswith("(") and t.endswith(")"):
            parts = [p.strip() for p in t[1:-1].split(",")]
            if len(parts) == 3:
                lookup[f"{parts[0]}|{parts[1]}|{parts[2]}"] = conf
    print(f"Agent confidence lookup: {len(lookup)} triples")
    return lookup

agent_conf_lookup = load_agent_confidence(agent_results)

# ════════════════════════════════════════════════════════
# STEP 2 — ENTITY PROFILES FOR NEGATIVE SAMPLING
# These replace jaccard_overlap (Nations) with
# type-constraint profile similarity (UMLS)
# ════════════════════════════════════════════════════════

# relational profile: what relations does each entity appear in?
entity_rel_profile = defaultdict(set)
for _, row in df_train.iterrows():
    entity_rel_profile[row["tail"]].add(row["relation"])
    entity_rel_profile[row["head"]].add(row["relation"])

# KGE score lookup: (head, relation, entity) → score
# batch score all entities for each query relation
def get_kge_scores_for_query(head, relation, model,
                              batch_size=256):
    """Score all entities as candidates for (head, relation, ?)"""
    h_id    = entity_to_id[head]
    r_id    = relation_to_id[relation]
    all_ids = list(range(len(entity_to_id)))

    scores = {}
    for i in range(0, len(all_ids), batch_size):
        batch = all_ids[i:i + batch_size]
        hrs   = torch.tensor([[h_id, r_id, t] for t in batch])
        with torch.no_grad():
            s = winner_model.score_hrt(hrs).squeeze(-1).cpu().tolist()
        for t_id, score in zip(batch, s):
            scores[id_to_entity[t_id]] = score
    return scores

def profile_similarity(e1, e2):
    """Jaccard on relational profiles — UMLS replacement for jaccard_overlap"""
    p1 = entity_rel_profile.get(e1, set())
    p2 = entity_rel_profile.get(e2, set())
    if not p1 or not p2:
        return 0.0
    return len(p1 & p2) / len(p1 | p2)

def type_fit(entity, relation):
    dist = constraints["rel_to_tail_dist"].get(relation, {})
    return dist.get(entity, 0.0)

# ════════════════════════════════════════════════════════
# STEP 3 — TEXT INPUT BUILDER (UMLS format)
# Must match exactly what agents saw during training
# ════════════════════════════════════════════════════════

def build_text_input_umls(head, relation, candidate,
                           kge_score, true_tail=None):
    """
    UMLS text_input format.
    Mirrors Nations [QUERY]/[CANDIDATE]/[KEY SIGNALS]/[DIFFERENCE]
    but replaces path signals with type constraint signals.
    """
    sig       = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = sig.get(candidate, 0.0)
    ranked    = list(sig.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    expected  = ", ".join(ranked[:3]) if ranked else "none"

    cand_rels = sorted(entity_rel_profile.get(candidate, set()))[:5]

    text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{candidate}\n\n"
        f"[TYPE CONSTRAINTS]\n"
        f"type_fit = {tf:.4f}\n"
        f"type_rank = {type_rank}\n"
        f"expected_tails = {expected}\n\n"
        f"[KGE SIGNALS]\n"
        f"kge_score = {kge_score:.3f}\n\n"
        f"[RELATIONAL PROFILE]\n"
        f"candidate_appears_in = "
        f"{', '.join(cand_rels) if cand_rels else 'none'}\n"
    )

    return text

# ════════════════════════════════════════════════════════
# STEP 4 — NEGATIVE SAMPLING (UMLS version)
#
# Nations had 4 neg_types: model_topk, structural,
#                          similarity, random
# UMLS replaces structural with type_confusable:
#   entities that have similar type_fit for this relation
#   i.e. they are plausible but wrong type slot fillers
# ════════════════════════════════════════════════════════

def sample_negatives_umls(head, relation, true_tail,
                           kge_scores, n_model=3,
                           n_type=2, n_random=1):
    """
    Three negative types for UMLS:

    model_topk:     entities RotatE ranked highly but wrong
                    hardest — model already confident about these

    type_confusable: entities with non-zero type_fit for this
                    relation but are not the true tail
                    medium hard — plausible type slot fillers

    random:         random entities
                    easy — provides diversity, prevents collapse
    """
    negatives  = []
    seen       = {true_tail, head}

    # ── model_topk: top RotatE scores excluding true tail ──
    model_ranked = sorted(
        [(e, s) for e, s in kge_scores.items() if e not in seen],
        key=lambda x: -x[1]
    )
    for entity, score in model_ranked[:n_model]:
        prof_sim = profile_similarity(entity, true_tail)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "model_topk",
            "features": {
                "kge_score":       score,
                "type_fit":        type_fit(entity, relation),
                "profile_sim":     round(prof_sim, 4),
                "jaccard_overlap": round(prof_sim, 4),  # compat alias
            }
        })
        seen.add(entity)

    # ── type_confusable: non-zero type_fit, not top-K ─────
    rel_dist = constraints["rel_to_tail_dist"].get(relation, {})
    type_candidates = [
        e for e in rel_dist
        if e not in seen and rel_dist[e] > 0.0
    ]
    # sort by type_fit descending — most confusable first
    type_candidates.sort(key=lambda e: -rel_dist[e])

    for entity in type_candidates[:n_type]:
        score    = kge_scores.get(entity, -99.0)
        prof_sim = profile_similarity(entity, true_tail)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "type_confusable",
            "features": {
                "kge_score":       score,
                "type_fit":        rel_dist[entity],
                "profile_sim":     round(prof_sim, 4),
                "jaccard_overlap": round(prof_sim, 4),
            }
        })
        seen.add(entity)

    # ── random: pure random sample ────────────────────────
    pool = [e for e in all_entities if e not in seen]
    for entity in random.sample(pool, min(n_random, len(pool))):
        score    = kge_scores.get(entity, -99.0)
        prof_sim = profile_similarity(entity, true_tail)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "random",
            "features": {
                "kge_score":       score,
                "type_fit":        type_fit(entity, relation),
                "profile_sim":     round(prof_sim, 4),
                "jaccard_overlap": round(prof_sim, 4),
            }
        })

    return negatives

# ════════════════════════════════════════════════════════
# STEP 5 — SOFT LABEL COMPUTATION (UMLS version)
#
# Nations used: kge_score + jaccard_overlap + agent_conf
# UMLS uses:    kge_score + profile_sim + type_fit + agent_conf
#
# type_fit replaces jaccard_overlap as the "how confusable"
# signal — entities with high type_fit for this relation
# are genuinely harder to distinguish from the true tail
# ════════════════════════════════════════════════════════

def compute_soft_labels_umls(candidates, agent_conf):
    """
    Soft label = how confusable is this negative with true tail?

    High soft_label → model should not be punished hard for
                      being confused (adaptive margin shrinks)
    Low soft_label  → easy negative, push hard

    Components:
      kge_norm:    how high did RotatE score this candidate?
                   (normalized within this query's negatives)
      type_norm:   how well does this candidate fit the type slot?
                   (replaces jaccard_overlap from Nations)
      agent_pen:   penalize based on agent confidence
                   low agent confidence → higher soft labels
                   (more uncertainty → be gentler)
    """
    negs = [c for c in candidates if c["label"] == 0]
    if not negs:
        return candidates

    kge_vals  = [c["features"].get("kge_score", -99.0) for c in negs]
    type_vals = [c["features"].get("type_fit", 0.0)    for c in negs]

    kge_min, kge_max   = min(kge_vals), max(kge_vals)
    type_min, type_max = min(type_vals), max(type_vals)
    kge_r  = max(kge_max  - kge_min,  1e-6)
    type_r = max(type_max - type_min, 1e-6)

    for c in candidates:
        if c["label"] == 1:
            c["soft_label"] = 1.0
            continue
        if c.get("neg_type") == "random":
            c["soft_label"] = 0.0
            continue

        kge_n  = (c["features"].get("kge_score", -99.0) - kge_min) / kge_r
        type_n = (c["features"].get("type_fit",   0.0)  - type_min) / type_r
        pen    = (1.0 - agent_conf) * W_AGENT

        soft = W_KGE * kge_n + W_OVERLAP * type_n + pen
        c["soft_label"] = round(min(max(soft, 0.0), 0.89), 4)

    return candidates

# ════════════════════════════════════════════════════════
# STEP 6 — MAIN BUILDER
# ════════════════════════════════════════════════════════

def build_umls_bert_dataset(agent_results, output_path,
                             n_model=3, n_type=2, n_random=1):

    bert_records = []
    skipped      = 0
    kge_cache    = {}   # cache scores per (head, relation)

    for i, result in enumerate(agent_results):

        if not result.get("final_correct"):
            continue
        if result.get("aggregator", {}).get("failure_type") != "resolved":
            continue

        # parse triple
        raw   = result["triple"].strip("()")
        parts = [p.strip() for p in raw.split(",")]
        if len(parts) != 3:
            skipped += 1
            continue
        head, relation, true_tail = parts

        # get preprocessed record
        pre = preprocessed_index.get((head, relation, true_tail))
        if pre is None:
            skipped += 1
            continue

        # get agent confidence
        agent_conf = agent_conf_lookup.get(
            f"{head}|{relation}|{true_tail}", 0.5
        )

        # get KGE scores (cached per head+relation)
        cache_key = (head, relation)
        if cache_key not in kge_cache:
            print(f"  Scoring KGE for {head} | {relation}...")
            kge_cache[cache_key] = get_kge_scores_for_query(
                head, relation, winner_model
            )
        kge_scores = kge_cache[cache_key]

        true_kge_score = kge_scores.get(true_tail, pre["score_true"])

        # ── positive candidate ────────────────────────────
        positive = {
            "entity":     true_tail,
            "label":      1,
            "soft_label": 1.0,
            "neg_type":   "positive",
            "features": {
                "kge_score":       true_kge_score,
                "type_fit":        type_fit(true_tail, relation),
                "profile_sim":     1.0,
                "jaccard_overlap": 1.0,
            },
            "text_input": build_text_input_umls(
                head, relation, true_tail,
                kge_score=true_kge_score,
                true_tail=None  # no [DIFFERENCE] for positive
            ),
        }

        # ── negative candidates ───────────────────────────
        negatives = sample_negatives_umls(
            head, relation, true_tail, kge_scores,
            n_model=n_model, n_type=n_type, n_random=n_random
        )

        # add text_input to each negative
        for neg in negatives:
            neg["text_input"] = build_text_input_umls(
                head, relation, neg["entity"],
                kge_score=neg["features"]["kge_score"],
                true_tail=true_tail   # adds [DIFFERENCE] block
            )

        # ── combine + soft labels ─────────────────────────
        all_candidates = [positive] + negatives
        all_candidates = compute_soft_labels_umls(
            all_candidates, agent_conf
        )
        all_candidates.sort(key=lambda c: -c["soft_label"])

        # ── only_tail_has from preprocessed ──────────────
        only_tail_has = pre.get("only_tail_has", [])

        bert_records.append({
            "triple":           [head, relation, true_tail],
            "true_rank":        pre["true_rank"],
            "hop_type":         pre.get("hop_type", "multi"),
            "hard_failure":     pre.get("hard_failure", False),
            "agent_confidence": agent_conf,
            "rotate_rank_in_topk": pre["true_rank"],
            "candidates":       [
                {
                    "entity":     c["entity"],
                    "label":      c["label"],
                    "soft_label": c["soft_label"],
                    "neg_type":   c.get("neg_type", "positive"),
                    "kge_rank":   sorted(
                        kge_scores.keys(),
                        key=lambda e: -kge_scores[e]
                    ).index(c["entity"]) + 1
                    if c["entity"] in kge_scores else 999,
                    "text_input": c["text_input"],
                    "features":   c.get("features", {}),
                }
                for c in all_candidates
            ],
            "context": {
                "subgraph_str":  pre.get("fail_summary", ""),
                "only_tail_has": only_tail_has,
                "fail_summary":  pre.get("fail_summary", ""),
            }
        })

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(agent_results)} done  "
                  f"({len(bert_records)} records built)")

    # ── save ──────────────────────────────────────────────
    with open(output_path, "w") as f:
        json.dump(bert_records, f, indent=2)

    # ── stats ─────────────────────────────────────────────
    all_soft = [
        c["soft_label"]
        for r in bert_records
        for c in r["candidates"]
        if c["label"] == 0
    ]
    neg_types = [
        c["neg_type"]
        for r in bert_records
        for c in r["candidates"]
        if c["label"] == 0
    ]
    from collections import Counter

    print(f"\n{'='*45}")
    print(f"BERT dataset built")
    print(f"  Records          : {len(bert_records)}")
    print(f"  Skipped          : {skipped}")
    print(f"  Avg candidates   : "
          f"{sum(len(r['candidates']) for r in bert_records)/max(len(bert_records),1):.1f}")
    print(f"\nSoft label distribution (negatives only):")
    print(f"  min  : {min(all_soft):.3f}")
    print(f"  max  : {max(all_soft):.3f}")
    print(f"  mean : {np.mean(all_soft):.3f}")
    print(f"\nNeg type breakdown:")
    for nt, count in Counter(neg_types).most_common():
        print(f"  {nt:<20} {count}")

    print(f"\nSample positive text_input:")
    print(bert_records[0]["candidates"][0]["text_input"])
    print(f"\nSample negative text_input (model_topk):")
    neg_ex = next(
        c for c in bert_records[0]["candidates"]
        if c["neg_type"] == "model_topk"
    )
    print(neg_ex["text_input"])

    return bert_records


# ── run ───────────────────────────────────────────────────
all_entities = list(entity_to_id.keys())
random.seed(42)

bert_data = build_umls_bert_dataset(
    agent_results = agent_results,
    output_path   = "umls_bert_reranker_train.json",
    n_model       = 3,
    n_type        = 2,
    n_random      = 1,
)


# ════════════════════════════════════════════════════════
# DATASET CLASS — plug into existing BERT training loop
# ════════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

class UMLSRerankerDataset(Dataset):
    """
    Same interface as BGERerankerDataset from Nations.
    Drop-in replacement — training loop unchanged.
    """

    def __init__(self, data, tokenizer,
                 max_length=MAX_LENGTH,
                 hard_k=2, rand_k=2, seed=None):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.hard_k    = hard_k
        self.rand_k    = rand_k
        self.seed      = seed
        self.pairs     = []
        self.resample()

    def resample(self):
        rng = random.Random(self.seed)
        self.pairs = []

        for ex in self.data:
            pos = next(
                (c for c in ex["candidates"] if c["label"] == 1),
                None
            )
            if pos is None:
                continue

            neg_pool = [c for c in ex["candidates"] if c["label"] == 0]

            # hard_k: highest soft_label first (most confusable)
            neg_pool.sort(key=lambda c: -c["soft_label"])
            selected  = neg_pool[:self.hard_k]
            remainder = neg_pool[self.hard_k:]

            # rand_k: random from remainder for diversity
            if remainder and self.rand_k > 0:
                k = min(self.rand_k, len(remainder))
                selected += rng.sample(remainder, k)

            for neg in selected:
                self.pairs.append({
                    "pos_text":   pos["text_input"],
                    "neg_text":   neg["text_input"],
                    "soft_label": float(neg["soft_label"]),
                    "neg_type":   neg.get("neg_type", "unknown"),
                })

        print(f"Pairs built: {len(self.pairs)}")
        self._print_stats()

    def _print_stats(self):
        from collections import Counter, defaultdict
        import numpy as np
        by_type = defaultdict(list)
        for p in self.pairs:
            by_type[p["neg_type"]].append(p["soft_label"])
        print("  Negative breakdown:")
        for nt, labels in sorted(by_type.items()):
            print(f"    {nt:<20} n={len(labels):>4}  "
                  f"avg_soft={np.mean(labels):.3f}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair    = self.pairs[idx]
        pos_enc = self.tokenizer(
            pair["pos_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        neg_enc = self.tokenizer(
            pair["neg_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "pos_input_ids":      pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids":      neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
            "soft_label":         torch.tensor(pair["soft_label"],
                                               dtype=torch.float32),
            "neg_type":           pair["neg_type"],
        }


# verify dataset builds correctly
dataset = UMLSRerankerDataset(
    bert_data, tokenizer, hard_k=2, rand_k=2, seed=42
)

pair = dataset[0]
print("\nPos decoded:")
print(tokenizer.decode(pair["pos_input_ids"],
                        skip_special_tokens=True)[:300])
print("\nNeg decoded:")
print(tokenizer.decode(pair["neg_input_ids"],
                        skip_special_tokens=True)[:300])
print("\nSoft label:", pair["soft_label"].item())
print("Neg type:  ", pair["neg_type"])

Agent results  : 100
Train records  : 5216
Test records   : 661
Index size     : 5877
Agent confidence lookup: 100 triples
  Scoring KGE for mental_or_behavioral_dysfunction | affects...
  Scoring KGE for tissue | issue_in...
  Scoring KGE for acquired_abnormality | result_of...
  Scoring KGE for mental_or_behavioral_dysfunction | process_of...
  Scoring KGE for organism_function | affects...
  Scoring KGE for fully_formed_anatomical_structure | location_of...
  Scoring KGE for laboratory_procedure | assesses_effect_of...
  Scoring KGE for physiologic_function | affects...
  Scoring KGE for experimental_model_of_disease | result_of...
  Scoring KGE for therapeutic_or_preventive_procedure | affects...
  10/100 done  (10 records built)
  Scoring KGE for pathologic_function | result_of...
  Scoring KGE for biologic_function | affects...
  Scoring KGE for chemical_viewed_structurally | interacts_with...
  Scoring KGE for molecular_biology_research_technique | measures...
  Scoring KGE for 

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Pairs built: 376
  Negative breakdown:
    model_topk           n= 221  avg_soft=0.652
    random               n=  40  avg_soft=0.000
    type_confusable      n= 115  avg_soft=0.560

Pos decoded:
[ query ] mental _ or _ behavioral _ dysfunction | affects |? [ candidate ] plant [ type constraints ] type _ fit = 0. 0149 type _ rank = 29 expected _ tails = cell _ function, molecular _ function, organism _ function [ kge signals ] kge _ score = - 8. 243 [ relational profile ] candidate _ appears

Neg decoded:
[ query ] mental _ or _ behavioral _ dysfunction | affects |? [ candidate ] molecular _ function [ type constraints ] type _ fit = 0. 0523 type _ rank = 2 expected _ tails = cell _ function, molecular _ function, organism _ function [ kge signals ] kge _ score = - 6. 987 [ relational profile ] candi

Soft label: 0.7900000214576721
Neg type:   model_topk


In [14]:
with open("umls_bert_reranker_train.json") as f:
    data = json.load(f)

total_pairs = sum(
    len([c for c in r["candidates"] if c["label"] == 0])
    for r in data
)
print(f"Records          : {len(data)}")
print(f"Total neg pairs  : {total_pairs}")
print(f"Avg negs/triple  : {total_pairs/len(data):.1f}")

# with 0.3 split:
n_val   = int(len(data) * 0.3)
n_train = len(data) - n_val
avg_neg = total_pairs / len(data)
print(f"\nWith 0.3 split:")
print(f"  Train triples : {n_train}")
print(f"  Val triples   : {n_val}")
print(f"  Train pairs   : ~{int(n_train * avg_neg)}")
print(f"  Val pairs     : ~{int(n_val * avg_neg)}")

Records          : 94
Total neg pairs  : 558
Avg negs/triple  : 5.9

With 0.3 split:
  Train triples : 66
  Val triples   : 28
  Train pairs   : ~391
  Val pairs     : ~166


In [15]:
# check how many train hard failures you have
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_preprocessed_train (2).json") as f:
    train_data = json.load(f)

n_entities     = len(entity_to_id)
hard_threshold = max(10, int(n_entities * 0.15))

train_hard = [r for r in train_data if r["hard_failure"]]
print(f"Train hard failures: {len(train_hard)}")
print(f"Test hard failures already run: 100")
print(f"Test hard failures remaining: ~75")
print(f"Total available: ~{len(train_hard) + 175}")

Train hard failures: 251
Test hard failures already run: 100
Test hard failures remaining: ~75
Total available: ~426


In [16]:
""""
import json, numpy as np, torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from collections import defaultdict

# ── config ────────────────────────────────────────────────
MAX_K        = 10
NEG_PRIORITY = ["model", "structural", "similarity", "random"]
W_KGE        = 0.5
W_AGENT      = 0.3
W_OVERLAP    = 0.2
MAX_LENGTH   = 256    # increased from 128 — structured format is longer

FEATURE_KEYS = [
    "kge_score", "sim_to_head", "sim_to_true_tail",
    "shared_rels_count", "only_cand_count", "only_true_count",
    "jaccard_overlap", "direct_edge", "hop_count",
]

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-large")



def load_agent_confidence(path=r"/kaggle/input/datasets/aaryaupi/bert-dataset/val_hard_results.json"):
    try:
        with open(path) as f:
            results = json.load(f)
    except FileNotFoundError:
        print(f"[WARN] {path} not found — defaulting conf=0.5")
        return {}

    lookup = {}
    for r in results:
        agg    = r.get("aggregator", {})
        chosen = agg.get("chosen_agent", "B")
        key_out = r.get("agent_a" if chosen == "A" else "agent_b", {})
        conf   = float(key_out.get("confidence", 0.5))

        t = r.get("triple", "")
        if t.startswith("(") and t.endswith(")"):
            parts = t[1:-1].split(", ")
            if len(parts) == 3:
                lookup[f"{parts[0]}|{parts[1]}|{parts[2]}"] = conf

    print(f"Agent confidence lookup: {len(lookup)} triples")
    return lookup


agent_conf_lookup = load_agent_confidence()

def format_candidate_structured(head:          str,
                                 relation:      str,
                                 candidate:     str,
                                 features:      dict,
                                 only_tail_has: list) -> str:
    """
    Replaces build_bert_text() from doc 5.
    
    Key difference: structured blocks instead of prose.
    Every field is labeled — no ambiguity for BERT.
    
    [QUERY]
    brazil | blockpositionindex | ?

    [CANDIDATE]
    china

    [KEY SIGNALS]
    kge_score = -5.327
    sim_to_head = 0.90
    shared_rels = 5
    only_true = 1
    only_cand = 0
    direct_edge = 1

    [DIFFERENCE]
    only_true_has = negativebehavior
    """
    f    = features
    diff = " | ".join(only_tail_has) if only_tail_has else "none"

    query_block     = f"[QUERY]\n{head} | {relation} | ?"
    candidate_block = f"[CANDIDATE]\n{candidate}"
    signals = (
        f"[KEY SIGNALS]\n"
        f"kge_score = {f.get('kge_score', 0.0):.3f}\n"
        f"sim_to_head = {f.get('sim_to_head', 0.0):.2f}\n"
        f"shared_rels = {f.get('shared_rels_count', 0)}\n"
        f"only_true = {f.get('only_true_count', 0)}\n"
        f"only_cand = {f.get('only_cand_count', 0)}\n"
        f"direct_edge = {f.get('direct_edge', 0)}"
    )
    diff_block = f"[DIFFERENCE]\nonly_true_has = {diff}"

    return "\n\n".join([query_block, candidate_block,
                        signals, diff_block])


# ════════════════════════════════════════════════════════
# STEP 3 — SOFT LABELS (unchanged from doc 5)
# ════════════════════════════════════════════════════════

def compute_soft_labels(candidates: list, agent_conf: float) -> list:
    neg_kge = [c["features"].get("kge_score", 0.0)
               for c in candidates if c["label"] == 0]
    neg_ov  = [c["features"].get("jaccard_overlap", 0.0)
               for c in candidates if c["label"] == 0]

    kge_min, kge_max = (min(neg_kge), max(neg_kge)) if neg_kge else (0.0, 1.0)
    ov_min,  ov_max  = (min(neg_ov),  max(neg_ov))  if neg_ov  else (0.0, 1.0)
    kge_r = max(kge_max - kge_min, 1e-6)
    ov_r  = max(ov_max  - ov_min,  1e-6)

    for c in candidates:
        if c["label"] == 1:
            c["soft_label"] = 1.0
        elif c.get("neg_type") == "random":
            c["soft_label"] = 0.0
        else:
            nk      = (c["features"].get("kge_score", 0.0)
                       - kge_min) / kge_r
            no      = (c["features"].get("jaccard_overlap", 0.0)
                       - ov_min) / ov_r
            penalty = (1.0 - agent_conf) * W_AGENT
            soft    = W_KGE * nk + penalty + W_OVERLAP * no
            c["soft_label"] = round(min(max(soft, 0.0), 0.89), 4)
    return candidates


# ════════════════════════════════════════════════════════
# STEP 4 — CANDIDATE SELECTION (unchanged from doc 5)
# ════════════════════════════════════════════════════════

def select_candidates(positive: dict,
                      negatives: list,
                      k: int = MAX_K) -> list:
    selected = [positive]
    seen     = {positive["entity"]}
    pool     = defaultdict(list)

    for neg in negatives:
        pool[neg.get("neg_type", "random")].append(neg)

    for neg_type in NEG_PRIORITY:
        for neg in pool[neg_type]:
            if neg["entity"] not in seen:
                selected.append(neg)
                seen.add(neg["entity"])
                if len(selected) == k:
                    return selected
    return selected


# ════════════════════════════════════════════════════════
# STEP 5 — CONVERT ONE RECORD
# Key change: text_input now uses format_candidate_structured
# instead of build_bert_text, and [DIFFERENCE] is recomputed
# per candidate (not shared from record level)
# ════════════════════════════════════════════════════════

def convert_record(record: dict) -> dict:
    head, relation, true_tail = record["triple"]
    agent_conf = agent_conf_lookup.get(
        f"{head}|{relation}|{true_tail}", 0.5
    )

    raw           = select_candidates(record["positive"],
                                      record["negatives"])
    only_tail_has = record.get("only_tail_has", [])

    for c in raw:
        if c["label"] == 1:
            # positive: full only_tail_has signal
            tail_has = only_tail_has
        else:
            # negative: recompute relative to THIS candidate
            # so [DIFFERENCE] reflects what THIS candidate lacks
            cand_shared = set(c.get("shared_relations", []))
            tail_has = [
                r for r in only_tail_has
                if r not in cand_shared
            ]

        # structured format replaces build_bert_text
        c["text_input"] = format_candidate_structured(
            head, relation, c["entity"],
            features      = c.get("features", {}),
            only_tail_has = tail_has,
        )

    candidates = compute_soft_labels(raw, agent_conf)
    candidates.sort(key=lambda c: -c["soft_label"])

    return {
        "triple":           [head, relation, true_tail],
        "true_rank":        record["true_rank"],
        "hop_type":         record.get("hop_type", "multi"),
        "agent_confidence": agent_conf,
        "candidates": [
            {
                "entity":     c["entity"],
                "label":      c["label"],
                "soft_label": c["soft_label"],
                "neg_type":   c.get("neg_type", "positive"),
                "text_input": c["text_input"],
                "features":   c.get("features", {}),
            }
            for c in candidates
        ],
        "context": {
            "subgraph_str":  record.get("subgraph_str", ""),
            "only_tail_has": only_tail_has,
            "fail_summary":  record.get("fail_summary", ""),
        }
    }




def build_reranker_json(input_path:  str = "bert_training_data.json",
                        output_path: str = "bert_reranker_train.json"):

    with open(input_path) as f:
        bert_data = json.load(f)

    print(f"Input records: {len(bert_data)}")

    reranker_data, errors = [], 0
    for i, record in enumerate(bert_data):
        try:
            reranker_data.append(convert_record(record))
        except Exception as e:
            print(f"[ERROR] {i}: {e}")
            errors += 1

    with open(output_path, "w") as f:
        json.dump(reranker_data, f, indent=2)

    # ── stats ──────────────────────────────────────────────
    all_soft = [
        c["soft_label"]
        for r in reranker_data
        for c in r["candidates"]
    ]
    print(f"\nOutput:  {len(reranker_data)} records  ({errors} errors)")
    print(f"Avg candidates per triple: "
          f"{sum(len(r['candidates']) for r in reranker_data) / max(len(reranker_data),1):.1f}")
    print(f"Soft label — min={min(all_soft):.3f}  "
          f"max={max(all_soft):.3f}  "
          f"mean={np.mean(all_soft):.3f}")
    print(f"\nSample text_input (positive):")
    print(reranker_data[0]["candidates"][0]["text_input"])
    print(f"\nSample text_input (first negative):")
    print(reranker_data[0]["candidates"][1]["text_input"])

    return reranker_data



class KGRerankerDataset(Dataset):
    """
    Reads from bert_reranker_train.json.
    Each __getitem__ = one (positive, negative) pair.

    text_input is already structured — just tokenize it.
    soft_label is used as pair weight during loss computation.
    """

    def __init__(self, data: list, max_length: int = MAX_LENGTH):
        self.pairs      = []
        self.max_length = max_length

        for ex in data:
            # find the positive candidate
            pos = next(
                (c for c in ex["candidates"] if c["label"] == 1),
                None
            )
            if pos is None:
                continue

            # pair positive against every negative
            for neg in ex["candidates"]:
                if neg["label"] == 1:
                    continue

                self.pairs.append({
                    "pos_text":   pos["text_input"],
                    "neg_text":   neg["text_input"],
                    "pos_feats":  self._get_features(pos),
                    "neg_feats":  self._get_features(neg),
                    "neg_type":   neg["neg_type"],
                    "soft_label": neg["soft_label"],  # pair weight
                    "true_rank":  ex["true_rank"],
                })

        print(f"Dataset: {len(self.pairs)} pairs  "
              f"({len(data)} triples)")

    def _get_features(self, candidate: dict) -> torch.Tensor:
        f = candidate.get("features", {})
        return torch.tensor(
            [float(f.get(k, 0.0)) for k in FEATURE_KEYS],
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict:
        pair = self.pairs[idx]

        pos_enc = tokenizer(
            pair["pos_text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        neg_enc = tokenizer(
            pair["neg_text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "pos_input_ids":      pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids":      neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
            "pos_features":       pair["pos_feats"],
            "neg_features":       pair["neg_feats"],
            "neg_type":           pair["neg_type"],
            "soft_label":         torch.tensor(pair["soft_label"],
                                               dtype=torch.float32),
        }


reranker_data = build_reranker_json(
    input_path  = r"/kaggle/input/datasets/aaryaupi/bert-dataset/bert_training_data.json",
    output_path = "bert_reranker_train.json",
)

dataset = KGRerankerDataset(reranker_data)

pair = dataset[0]
print("\nPos tokens decoded:")
print(tokenizer.decode(pair["pos_input_ids"], skip_special_tokens=True)[:300])
print("\nNeg tokens decoded:")
print(tokenizer.decode(pair["neg_input_ids"], skip_special_tokens=True)[:300])
print("\nSoft label:", pair["soft_label"].item())
print("Neg type:",   pair["neg_type"])
"""

IndentationError: unexpected indent (3022009226.py, line 58)

In [17]:
# DIAGNOSIS — run this before continuing training

import json
import numpy as np

with open(r"/kaggle/working/umls_bert_reranker_train.json") as f:
    data = json.load(f)

# split the same way your code does
import random
random.seed(42)
random.shuffle(data)
n_val    = max(1, int(len(data) * 0.1))
val_data = data[:n_val]

# check 1 — are val negatives too easy?
val_soft = []
for ex in val_data:
    for c in ex["candidates"]:
        if c["label"] == 0:
            val_soft.append(c["soft_label"])

print(f"Val neg soft_label distribution:")
print(f"  mean  = {np.mean(val_soft):.3f}")
print(f"  min   = {np.min(val_soft):.3f}")
print(f"  max   = {np.max(val_soft):.3f}")
print(f"  >0.7  = {sum(1 for s in val_soft if s > 0.7)} / {len(val_soft)}")
print(f"  =0.0  = {sum(1 for s in val_soft if s == 0.0)} / {len(val_soft)}")

# check 2 — is val leaking training triples?
val_triples  = set(tuple(ex["triple"]) for ex in val_data)
train_data   = data[n_val:]
train_triples = set(tuple(ex["triple"]) for ex in train_data)
overlap = val_triples & train_triples
print(f"\nVal/train triple overlap: {len(overlap)} / {len(val_triples)}")
# if this is > 0, your val is contaminated

# check 3 — kge score gap between pos and neg in val
print(f"\nKGE score gap in val (pos_kge - neg_kge):")
gaps = []
for ex in val_data:
    pos = next((c for c in ex["candidates"] if c["label"] == 1), None)
    if pos is None:
        continue
    for neg in ex["candidates"]:
        if neg["label"] == 0:
            gap = pos["features"]["kge_score"] - neg["features"]["kge_score"]
            gaps.append(gap)
print(f"  mean gap = {np.mean(gaps):.3f}")
print(f"  gap > 0  = {sum(1 for g in gaps if g > 0)} / {len(gaps)}")
# if gap > 0 most of the time, KGE already separates val easily
# model just learned to follow KGE score, not structural signal

Val neg soft_label distribution:
  mean  = 0.529
  min   = 0.000
  max   = 0.802
  >0.7  = 11 / 53
  =0.0  = 9 / 53

Val/train triple overlap: 0 / 9

KGE score gap in val (pos_kge - neg_kge):
  mean gap = -0.830
  gap > 0  = 9 / 53


In [18]:
# ════════════════════════════════════════════════════════
# AGENT C — BGE RERANKER TRAINING (MERGED)
# Adaptive margin loss + staged unfreezing + resampling
# ════════════════════════════════════════════════════════

import json, os, random, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from collections import defaultdict
import numpy as np

# ── config ────────────────────────────────────────────────
MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Try in order:
# 1. "BAAI/bge-reranker-base"                    — best, may need HF token
# 2. "cross-encoder/ms-marco-MiniLM-L-6-v2"     — always works, no auth
# 3. "cross-encoder/ms-marco-MiniLM-L-2-v2"     — lightest

MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS     = 4
LR         = 2e-5
MARGIN     = 1.0
SAVE_DIR   = "agent_c_bge"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ════════════════════════════════════════════════════════
# MODEL LOADING
# ════════════════════════════════════════════════════════

def load_model_and_tokenizer(model_name: str):
    """
    Tries BAAI first, falls back gracefully.
    BAAI/bge-reranker-base sometimes needs:
      pip install -U FlagEmbedding
    or HF login:
      huggingface-cli login
    """
    try:
        print(f"Loading {model_name}...")
        tok = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True
        )
        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1,
            trust_remote_code=True,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        print(f"Loaded: {model_name}")
        return mdl, tok

    except Exception as e:
        fallback = "cross-encoder/ms-marco-MiniLM-L-6-v2"
        print(f"[WARN] {model_name} failed: {e}")
        print(f"[WARN] Falling back to {fallback}")
        tok = AutoTokenizer.from_pretrained(fallback)
        mdl = AutoModelForSequenceClassification.from_pretrained(
            fallback, num_labels=1
        ).to(DEVICE)
        return mdl, tok


# ════════════════════════════════════════════════════════
# DATASET — supports resampling for staged training
#
# hard_k  : top-k hardest negatives by soft_label (descending)
# rand_k  : additional random negatives for diversity
# seed    : fixed seed locks val set; None = fresh draw each resample()
# ════════════════════════════════════════════════════════

class BGERerankerDataset(Dataset):

    def __init__(self, data: list, tokenizer,
                 max_length: int = MAX_LENGTH,
                 hard_k: int = 2, rand_k: int = 2,
                 seed: int | None = None):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.hard_k    = hard_k
        self.rand_k    = rand_k
        self.seed      = seed
        self.pairs     = []
        self.resample()

    def resample(self):
        """Rebuild pairs — call once per epoch on train_ds for fresh negatives."""
        rng = random.Random(self.seed)
        self.pairs = []

        for ex in self.data:
            pos = next(
                (c for c in ex["candidates"] if c["label"] == 1), None
            )
            if pos is None:
                continue

            neg_pool = [c for c in ex["candidates"] if c["label"] == 0]
            neg_pool.sort(key=lambda c: -c["soft_label"])

            selected = neg_pool[:self.hard_k]
            remainder = neg_pool[self.hard_k:]
            if remainder and self.rand_k > 0:
                k = min(self.rand_k, len(remainder))
                selected += rng.sample(remainder, k)

            for neg in selected:
                self.pairs.append({
                    "pos_text":   pos["text_input"],
                    "neg_text":   neg["text_input"],
                    "soft_label": float(neg["soft_label"]),
                    "neg_type":   neg.get("neg_type", "unknown"),
                    "true_rank":  ex.get("true_rank", 99),
                })

        print(f"Pairs built: {len(self.pairs)}")
        self._print_stats()

    def _print_stats(self):
        by_type = defaultdict(list)
        for p in self.pairs:
            by_type[p["neg_type"]].append(p["soft_label"])
        print("  Negative breakdown:")
        for nt, labels in sorted(by_type.items()):
            print(f"    {nt:<12} n={len(labels):>4}  "
                  f"avg_soft={np.mean(labels):.3f}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict:
        pair    = self.pairs[idx]
        pos_enc = self.tokenizer(
            pair["pos_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        neg_enc = self.tokenizer(
            pair["neg_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "pos_input_ids":      pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids":      neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
            "soft_label":         torch.tensor(pair["soft_label"],
                                               dtype=torch.float32),
            "neg_type":           pair["neg_type"],
        }


# ════════════════════════════════════════════════════════
# LOSS — adaptive margin + correct weighting + MSE aux
#
# FIX 1 — adaptive margin
#   margin shrinks as soft_label rises
#   soft=0.85 (hard/ambiguous) → margin=0.15  gentle push
#   soft=0.00 (easy)           → margin=1.0   full push
#
# FIX 2 — correct weighting
#   weight = 1 + (1 - soft_label)
#   easy negatives (soft=0)    → weight=2.0   push hard
#   hard negatives (soft=0.85) → weight=1.15  push gently
#
# FIX 3 — MSE auxiliary loss
#   pred_diff = score_pos - score_neg should equal (1 - soft)
#   anchors calibration so scores are meaningful, not just ordinal
#
# Doc-1 alias: FullDistillationLoss → same contract, kept as AdaptiveSoftLoss
# ════════════════════════════════════════════════════════

class AdaptiveSoftLoss(nn.Module):
    """
    Replaces FullDistillationLoss from Doc-1.
    Signature-compatible: forward() returns (total, pairwise, mse).
    Doc-1 used w_mse / w_abs; here mse_weight covers both.
    """

    def __init__(self, margin: float = MARGIN, mse_weight: float = 0.3,
                 # Accept but ignore Doc-1 kwargs for drop-in compatibility
                 w_mse: float | None = None, w_abs: float | None = None):
        super().__init__()
        self.margin     = margin
        # If called with Doc-1's w_mse, honour it
        self.mse_weight = w_mse if w_mse is not None else mse_weight

    def forward(self,
                score_pos:   torch.Tensor,
                score_neg:   torch.Tensor,
                soft_labels: torch.Tensor,
                ) -> tuple[torch.Tensor, float, float]:

        adaptive_margin = (1.0 - soft_labels) * self.margin
        raw_loss = torch.clamp(
            adaptive_margin - (score_pos - score_neg), min=0.0
        )
        weights       = 1.0 + (1.0 - soft_labels)
        pairwise_loss = (raw_loss * weights).mean()

        pred_diff   = score_pos - score_neg
        target_diff = 1.0 - soft_labels
        mse_loss    = nn.functional.mse_loss(pred_diff, target_diff)

        total = pairwise_loss + self.mse_weight * mse_loss
        return total, pairwise_loss.item(), mse_loss.item()

# Alias so old imports / references still resolve
FullDistillationLoss = AdaptiveSoftLoss


# ════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════

def compute_metrics(score_pos: torch.Tensor,
                    score_neg: torch.Tensor,
                    neg_types: list) -> dict:
    correct = (score_pos > score_neg).cpu()
    overall = correct.float().mean().item()

    by_type = defaultdict(lambda: {"correct": 0, "total": 0})
    for c, nt in zip(correct.tolist(), neg_types):
        by_type[nt]["correct"] += int(c)
        by_type[nt]["total"]   += 1

    return {
        "overall": overall,
        "by_type": {
            nt: v["correct"] / v["total"]
            for nt, v in by_type.items()
        },
    }


# ════════════════════════════════════════════════════════
# TRAIN EPOCH
# ════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer,
                scheduler, loss_fn, device) -> dict:
    model.train()
    total_loss = total_pair = total_mse = 0.0
    all_pos, all_neg, types = [], [], []

    for batch in loader:
        pos_ids  = batch["pos_input_ids"].to(device)
        pos_mask = batch["pos_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        soft     = batch["soft_label"].to(device)

        optimizer.zero_grad()
        score_pos = model(input_ids=pos_ids,
                          attention_mask=pos_mask).logits.squeeze(-1)
        score_neg = model(input_ids=neg_ids,
                          attention_mask=neg_mask).logits.squeeze(-1)

        loss, pair_l, mse_l = loss_fn(score_pos, score_neg, soft)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_pair += pair_l
        total_mse  += mse_l
        all_pos.extend(score_pos.detach().cpu().tolist())
        all_neg.extend(score_neg.detach().cpu().tolist())
        types.extend(batch["neg_type"])

    n = len(loader)
    metrics = compute_metrics(
        torch.tensor(all_pos), torch.tensor(all_neg), types
    )
    metrics["loss"]      = round(total_loss / n, 4)
    metrics["pairwise"]  = round(total_pair / n, 4)
    metrics["mse"]       = round(total_mse  / n, 4)
    metrics["abs"]       = metrics["mse"]   # kept for Doc-1 print compat
    return metrics


# ════════════════════════════════════════════════════════
# EVAL EPOCH
# ════════════════════════════════════════════════════════

@torch.no_grad()
def eval_epoch(model, loader, loss_fn, device) -> dict:
    model.eval()
    total_loss = total_pair = total_mse = 0.0
    all_pos, all_neg, types = [], [], []

    for batch in loader:
        pos_ids  = batch["pos_input_ids"].to(device)
        pos_mask = batch["pos_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        soft     = batch["soft_label"].to(device)

        score_pos = model(input_ids=pos_ids,
                          attention_mask=pos_mask).logits.squeeze(-1)
        score_neg = model(input_ids=neg_ids,
                          attention_mask=neg_mask).logits.squeeze(-1)

        loss, pair_l, mse_l = loss_fn(score_pos, score_neg, soft)
        total_loss += loss.item()
        total_pair += pair_l
        total_mse  += mse_l
        all_pos.extend(score_pos.cpu().tolist())
        all_neg.extend(score_neg.cpu().tolist())
        types.extend(batch["neg_type"])

    n = len(loader)
    metrics = compute_metrics(
        torch.tensor(all_pos), torch.tensor(all_neg), types
    )
    metrics["loss"]     = round(total_loss / n, 4)
    metrics["pairwise"] = round(total_pair / n, 4)
    metrics["mse"]      = round(total_mse  / n, 4)
    metrics["abs"]      = metrics["mse"]
    return metrics


# ════════════════════════════════════════════════════════
# TRAIN — staged unfreezing + per-epoch resampling
#
# Healthy convergence targets:
#   epoch 1 (head only):  train acc 0.75-0.85  val acc 0.85-0.92
#   epoch 2 (unfrozen):   train/val acc rising  mse still falling
#   epoch 3+:             train ~0.92           val ~0.88-0.92  mse < 0.15
#
# RED FLAGS:
#   val acc 0.99 in epoch 1  → val too easy
#   mse rising, pair falling → score gap overshooting
#   train acc >> val acc     → overfit on hard negatives
# ════════════════════════════════════════════════════════

def train(data_path: str = "bert_reranker_train.json"):

    with open(data_path) as f:
        raw = json.load(f)
    print(f"Records loaded: {len(raw)}")

    random.seed(42)
    random.shuffle(raw)
    n_val     = max(1, int(len(raw) * 0.1))
    val_raw   = raw[:n_val]
    train_raw = raw[n_val:]

    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)

    train_ds = BGERerankerDataset(
        train_raw, tokenizer, hard_k=2, rand_k=2, seed=None   # fresh each epoch
    )
    val_ds = BGERerankerDataset(
        val_raw, tokenizer, hard_k=2, rand_k=2, seed=42        # locked
    )

    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True,
    )

    loss_fn = AdaptiveSoftLoss(margin=MARGIN, w_mse=0.7, w_abs=0.2)

    os.makedirs(SAVE_DIR, exist_ok=True)
    best_val_acc = 0.0
    history      = []
    optimizer    = None   # assigned per-epoch below

    for epoch in range(1, EPOCHS + 1):

        # ── staged unfreezing ─────────────────────────────
        if epoch == 1:
            # Freeze BERT — only head trains.
            # Stops the head memorising trivial patterns before
            # BERT has adapted to the structured input format.
            for param in model.base_model.parameters():
                param.requires_grad = False
            print(f"\n── Epoch {epoch}: BERT frozen, training head only")
            optimizer = AdamW(
                model.classifier.parameters(),
                lr=LR * 10, weight_decay=0.01,
            )

        elif epoch == 2:
            # Unfreeze BERT at low LR; head keeps higher LR.
            for param in model.base_model.parameters():
                param.requires_grad = True
            print(f"\n── Epoch {epoch}: BERT unfrozen")
            optimizer = AdamW([
                {"params": model.base_model.parameters(),  "lr": LR},
                {"params": model.classifier.parameters(),  "lr": LR * 5},
            ], weight_decay=0.01)

        else:
            print(f"\n── Epoch {epoch}")

        # Rebuild pairs and loader each epoch (fresh random negatives)
        train_ds.resample()
        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE,
            shuffle=True, num_workers=2, pin_memory=True,
        )

        # Rebuild scheduler whenever optimizer changes (epochs 1 & 2)
        total_steps = len(train_loader) * (EPOCHS - epoch + 1)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = max(1, int(total_steps * 0.1)),
            num_training_steps = total_steps,
        )

        tr = train_epoch(model, train_loader, optimizer,
                         scheduler, loss_fn, DEVICE)
        vl = eval_epoch(model, val_loader, loss_fn, DEVICE)

        history.append({"epoch": epoch, "train": tr, "val": vl})

        print(f"  train  loss={tr['loss']}  pair={tr['pairwise']}  "
              f"mse={tr['mse']}  abs={tr['abs']}  acc={tr['overall']:.3f}")
        print(f"  val    loss={vl['loss']}  pair={vl['pairwise']}  "
              f"mse={vl['mse']}  abs={vl['abs']}  acc={vl['overall']:.3f}")
        print(f"  val breakdown:")
        for nt, acc in sorted(vl["by_type"].items()):
            bar = "█" * int(acc * 20)
            print(f"    {nt:<12} {acc:.3f}  {bar}")

        if vl["overall"] > best_val_acc:
            best_val_acc = vl["overall"]
            model.save_pretrained(SAVE_DIR)
            tokenizer.save_pretrained(SAVE_DIR)
            print(f"  ✓ saved (val_acc={best_val_acc:.3f})")

    with open(os.path.join(SAVE_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nDone. Best={best_val_acc:.3f} → {SAVE_DIR}/")
    return model, tokenizer, history


# ════════════════════════════════════════════════════════
model, tokenizer, history = train(r"/kaggle/working/umls_bert_reranker_train.json")

Records loaded: 94
Loading cross-encoder/ms-marco-MiniLM-L-6-v2...


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Pairs built: 340
  Negative breakdown:
    model_topk   n= 205  avg_soft=0.646
    random       n=  36  avg_soft=0.000
    type_confusable n=  99  avg_soft=0.580
Pairs built: 36
  Negative breakdown:
    model_topk   n=  22  avg_soft=0.675
    random       n=   3  avg_soft=0.000
    type_confusable n=  11  avg_soft=0.641

── Epoch 1: BERT frozen, training head only
Pairs built: 340
  Negative breakdown:
    model_topk   n= 196  avg_soft=0.647
    random       n=  48  avg_soft=0.000
    type_confusable n=  96  avg_soft=0.590
  train  loss=2.0582  pair=1.1449  mse=1.3048  abs=1.3048  acc=0.426
  val    loss=1.6513  pair=0.8543  mse=1.1386  abs=1.1386  acc=0.333
  val breakdown:
    model_topk   0.455  █████████
    random       0.333  ██████
    type_confusable 0.091  █


...

  ✓ saved (val_acc=0.806)

── Epoch 3
Pairs built: 340
  Negative breakdown:
    model_topk   n= 201  avg_soft=0.646
    random       n=  47  avg_soft=0.000
    type_confusable n=  92  avg_soft=0.576
  train  loss=0.8623  pair=0.6177  mse=0.3495  abs=0.3495  acc=0.641
  val    loss=0.2605  pair=0.1711  mse=0.1278  abs=0.1278  acc=0.917
  val breakdown:
    model_topk   0.909  ██████████████████
    random       1.000  ████████████████████
    type_confusable 0.909  ██████████████████


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ saved (val_acc=0.917)

── Epoch 4
Pairs built: 340
  Negative breakdown:
    model_topk   n= 201  avg_soft=0.648
    random       n=  47  avg_soft=0.000
    type_confusable n=  92  avg_soft=0.574
  train  loss=0.7394  pair=0.511  mse=0.3262  abs=0.3262  acc=0.762
  val    loss=0.1497  pair=0.0932  mse=0.0808  abs=0.0808  acc=1.000
  val breakdown:
    model_topk   1.000  ████████████████████
    random       1.000  ████████████████████
    type_confusable 1.000  ████████████████████


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ saved (val_acc=1.000)

Done. Best=1.000 → agent_c_bge/


In [19]:
import json, torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict
from tqdm import tqdm


RERANKER_DIR = "agent_c_bge"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH   = 512
BATCH_SIZE   = 32
ALPHA        = 0.4
DELTA        = 0.6
T            = 3

def load_reranker(path):
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        path, trust_remote_code=True
    ).to(DEVICE)
    mdl.eval()
    return mdl, tok


# ════════════════════════════════════════════════════════
# BERT SCORING (BATCHED)
# ════════════════════════════════════════════════════════

@torch.no_grad()
def bert_score_batch(model, tokenizer, texts):
    scores = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(-1)
        scores.extend(logits.cpu().tolist())
    return scores

def minmax(x):
    x = np.array(x, dtype=float)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-9:
        return np.full_like(x, 0.5)
    return (x - lo) / (hi - lo)


# ════════════════════════════════════════════════════════
# INJECT TRUE TAIL IF MISSING FROM CANDIDATES
# ════════════════════════════════════════════════════════

def ensure_true_tail(candidates, true_tail, head, relation):
    if true_tail in [c["entity"] for c in candidates]:
        return candidates, False

    worst_kge = min(
        c["features"]["kge_score"]
        for c in candidates
        if c.get("features", {}).get("kge_score") is not None
    )
    injected_text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{true_tail}\n\n"
        f"[KEY SIGNALS]\n(injected — not in top-K)"
    )
    candidates = candidates + [{
        "entity":     true_tail,
        "label":      1,
        "neg_type":   "true_tail_injected",
        "kge_rank":   len(candidates) + 1,
        "text_input": injected_text,
        "features":   {"kge_score": worst_kge - 0.5},
    }]
    return candidates, True


# ════════════════════════════════════════════════════════
# MAIN EVALUATION
# ════════════════════════════════════════════════════════

def evaluate(test_path, alpha=ALPHA, delta=DELTA, T=T):

    print("Loading reranker...")
    model, tokenizer = load_reranker(RERANKER_DIR)

    with open(test_path) as f:
        data = json.load(f)
    print(f"Queries: {len(data)}")

    results    = []
    n_injected = 0
    skipped    = 0

    for rec in tqdm(data):

        triple = rec["triple"]
        head, relation, tail = (
            [x.strip() for x in triple]
            if isinstance(triple, list)
            else [x.strip() for x in triple.split("|")]
        )
        true_tail = tail

        candidates = rec.get("candidates", [])
        if not candidates:
            skipped += 1
            continue

        candidates, injected = ensure_true_tail(
            candidates, true_tail, head, relation
        )
        if injected:
            n_injected += 1

        texts = [c["text_input"] for c in candidates]

        bert_scores = bert_score_batch(model, tokenizer, texts)

        kge_scores = [
            c["features"].get("kge_score") or (1.0 / max(c.get("kge_rank", 99), 1))
            for c in candidates
        ]

        kge_norm  = minmax(kge_scores)
        bert_norm = minmax(bert_scores)
        combined  = alpha * kge_norm + delta * bert_norm

        enriched = [
            {
                "entity":     c["entity"],
                "kge_score":  ks,
                "bert_score": bs,
                "combined":   float(cs),
                "neg_type":   c.get("neg_type", "unknown"),
            }
            for c, ks, bs, cs in zip(candidates, kge_scores, bert_scores, combined)
        ]

        ranked_comb = sorted(enriched, key=lambda x: -x["combined"])
        ranked_bert = sorted(enriched, key=lambda x: -x["bert_score"])
        ranked_kge  = sorted(enriched, key=lambda x: -x["kge_score"])

        def find_rank(ranked):
            for i, x in enumerate(ranked, 1):
                if x["entity"] == true_tail:
                    return i
            return len(ranked)

        rank_comb = find_rank(ranked_comb)
        rank_bert = find_rank(ranked_bert)
        rank_kge  = find_rank(ranked_kge)

        results.append({
            "triple":       f"{head}|{relation}|{true_tail}",
            "true_tail":    true_tail,
            "true_rank":    rec.get("true_rank"),
            "rotate_rank":  rec.get("rotate_rank_in_topk"),
            "hop_type":     rec.get("hop_type", "?"),
            "hard_failure": rec.get("hard_failure", False),
            "injected":     injected,
            "rank_kge":     rank_kge,
            "rank_bert":    rank_bert,
            "rank_comb":    rank_comb,
            "top3_comb":    [x["entity"] for x in ranked_comb[:3]],
        })

    if skipped:
        print(f"[WARN] skipped {skipped} records with no candidates")
    if n_injected:
        print(f"[INFO] injected true tail into {n_injected}/{len(data)} queries")

    # ════════════════════════════════════════════════════
    # METRICS
    # ════════════════════════════════════════════════════

    def compute(ranks):
        ranks = np.array(ranks)
        return {
            "Hits@1":  float(np.mean(ranks == 1)),
            "Hits@3":  float(np.mean(ranks <= 3)),
            "Hits@10": float(np.mean(ranks <= 10)),
            "MRR":     float(np.mean(1.0 / ranks)),
        }

    def print_metrics(label, m):
        print(f"\n{label}")
        for k in ["Hits@1", "Hits@3", "Hits@10", "MRR"]:
            print(f"  {k:<8}: {m[k]:.4f}")

    kge_m  = compute([r["rank_kge"]  for r in results])
    bert_m = compute([r["rank_bert"] for r in results])
    comb_m = compute([r["rank_comb"] for r in results])

    # ── lucky rate ────────────────────────────────────────
    hits1    = [r for r in results if r["rank_comb"] == 1]
    grounded = [r for r in hits1   if r["rank_bert"] <= T]
    lucky    = [r for r in hits1   if r["rank_bert"] >  T]
    n        = len(hits1)

    # ── by hop type ───────────────────────────────────────
    by_hop = defaultdict(list)
    for r in results:
        by_hop[r["hop_type"]].append(r["rank_comb"])

    # ── hard failure recovery ─────────────────────────────
    hard     = [r for r in results if r["hard_failure"]]
    not_hard = [r for r in results if not r["hard_failure"]]

    # ── vs rotate rank ────────────────────────────────────
    improved  = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] < r["rotate_rank"])
    worsened  = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] > r["rotate_rank"])
    unchanged = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] == r["rotate_rank"])

    # ════════════════════════════════════════════════════
    # PRINT
    # ════════════════════════════════════════════════════

    print("\n═══════════════════════════════════════════")
    print("FINAL RESULTS")
    print("═══════════════════════════════════════════")

    print_metrics("KGE  (RotatE baseline)", kge_m)
    print_metrics("BERT (reranker only)",   bert_m)
    print_metrics("COMBINED",               comb_m)

    print(f"\n── Lucky rate  (T={T}) ─────────────────────")
    if n:
        print(f"  Rank-1 queries : {n}")
        print(f"  Grounded       : {len(grounded):>4}  ({len(grounded)/n:.1%})  ← BERT agreed independently")
        print(f"  Lucky          : {len(lucky):>4}  ({len(lucky)/n:.1%})  ← KGE rescued it")
        verdict = "reranker drives wins ✓" if len(grounded) > len(lucky) else "KGE doing the work — reranker marginal"
        print(f"  Verdict        : {verdict}")
    else:
        print("  No Hits@1")

    print(f"\n── vs RotatE rank in top-K ─────────────────")
    print(f"  Improved  : {improved}")
    print(f"  Unchanged : {unchanged}")
    print(f"  Worsened  : {worsened}")

    print(f"\n── By hop type ─────────────────────────────")
    for ht, ranks in sorted(by_hop.items()):
        m = compute(ranks)
        bar = "█" * int(m["Hits@1"] * 20)
        print(f"  {ht:<8}  n={len(ranks):>4}  Hits@1={m['Hits@1']:.3f}  MRR={m['MRR']:.3f}  {bar}")

    if hard:
        hm = compute([r["rank_comb"] for r in hard])
        nm = compute([r["rank_comb"] for r in not_hard])
        print(f"\n── Hard failure recovery ────────────────────")
        print(f"  hard=True   n={len(hard):>4}  Hits@1={hm['Hits@1']:.3f}  MRR={hm['MRR']:.3f}")
        print(f"  hard=False  n={len(not_hard):>4}  Hits@1={nm['Hits@1']:.3f}  MRR={nm['MRR']:.3f}")

    # ════════════════════════════════════════════════════
    # SAVE
    # ════════════════════════════════════════════════════

    output = {
        "config":  {"alpha": alpha, "delta": delta, "T": T},
        "metrics": {"kge": kge_m, "bert": bert_m, "combined": comb_m},
        "lucky": {
            "grounded":    len(grounded),
            "lucky":       len(lucky),
            "total_hits1": n,
            "lucky_rate":  len(lucky)    / n if n else 0.0,
            "ground_rate": len(grounded) / n if n else 0.0,
        },
        "vs_rotate": {
            "improved": improved, "unchanged": unchanged, "worsened": worsened
        },
        "per_query": results,
    }
    with open("inference_results.json", "w") as f:
        json.dump(output, f, indent=2)
    print("\nSaved → inference_results.json")
    return output


# ════════════════════════════════════════════════════════
if __name__ == "__main__":
    evaluate(r"/kaggle/input/datasets/aaryaupi/umls2-8502/umls_bert_held_out (2).json")

Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Queries: 331


100%|██████████| 331/331 [00:18<00:00, 17.69it/s]


═══════════════════════════════════════════
FINAL RESULTS
═══════════════════════════════════════════

KGE  (RotatE baseline)
  Hits@1  : 0.0091
  Hits@3  : 0.0544
  Hits@10 : 0.3595
  MRR     : 0.1074

BERT (reranker only)
  Hits@1  : 0.0937
  Hits@3  : 0.1752
  Hits@10 : 0.3897
  MRR     : 0.1917

COMBINED
  Hits@1  : 0.0665
  Hits@3  : 0.1752
  Hits@10 : 0.4411
  MRR     : 0.1798

── Lucky rate  (T=3) ─────────────────────
  Rank-1 queries : 22
  Grounded       :   20  (90.9%)  ← BERT agreed independently
  Lucky          :    2  (9.1%)  ← KGE rescued it
  Verdict        : reranker drives wins ✓

── vs RotatE rank in top-K ─────────────────
  Improved  : 169
  Unchanged : 8
  Worsened  : 154

── By hop type ─────────────────────────────
  multi     n= 327  Hits@1=0.067  MRR=0.181  █
  none      n=   4  Hits@1=0.000  MRR=0.074  

── Hard failure recovery ────────────────────
  hard=True   n=  96  Hits@1=0.042  MRR=0.131
  hard=False  n= 235  Hits@1=0.077  MRR=0.200

Saved → inferenc

In [ ]:
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_bert_held_out (1).json") as f:
    data = json.load(f)

print("Top-level keys:", list(data[0].keys()))
print("\nFull first record (no candidates):")
rec = {k: v for k, v in data[0].items() if k != "candidates"}
print(json.dumps(rec, indent=2))
print("\nFirst candidate keys:", list(data[0]["candidates"][0].keys()) if data[0].get("candidates") else "no candidates key")

In [ ]:
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_bert_held_out (1).json") as f:
    data = json.load(f)

missing = 0
for rec in data:
    triple = rec["triple"]
    true_tail = triple[2].strip() if isinstance(triple, list) else triple.split("|")[2].strip()
    entities = [c["entity"] for c in rec.get("candidates", [])]
    if true_tail not in entities:
        missing += 1

print(f"True tail missing from candidates: {missing}/{len(data)}")

In [ ]:
with open(r"/kaggle/input/datasets/aaryaupi/umls-dataset/umls_bert_held_out (1).json") as f:
    data = json.load(f)

# Check true tail's position and neg_type in candidates
for i, rec in enumerate(data[:5]):
    triple = rec["triple"]
    true_tail = triple[2].strip() if isinstance(triple, list) else triple.split("|")[2].strip()
    for j, c in enumerate(rec["candidates"]):
        if c["entity"] == true_tail:
            print(f"Record {i}: true_tail='{true_tail}'  pos={j+1}/{len(rec['candidates'])}  label={c['label']}  neg_type={c['neg_type']}  kge_rank={c.get('kge_rank')}")
            print(f"  text_input preview: {c['text_input'][:120].strip()}")
            print()

In [ ]:
neg_types = []
kge_ranks = []
positions = []

for rec in data:
    triple = rec["triple"]
    true_tail = triple[2].strip() if isinstance(triple, list) else triple.split("|")[2].strip()
    entities = [c["entity"] for c in rec["candidates"]]
    pos = entities.index(true_tail)
    c = rec["candidates"][pos]
    neg_types.append(c["neg_type"])
    kge_ranks.append(c.get("kge_rank"))
    positions.append(pos + 1)

from collections import Counter
print("neg_type distribution for true tail:", Counter(neg_types))
print(f"mean position in candidate list: {sum(positions)/len(positions):.1f}")
print(f"always position 1: {sum(1 for p in positions if p == 1)}/{len(positions)}")
print(f"kge_rank=1 count: {sum(1 for k in kge_ranks if k == 1)}/{len(kge_ranks)}")

In [ ]:
# Find queries where the true tail has NO unique relations
hard_queries = []
for rec in data:
    for c in rec["candidates"]:
        if c["neg_type"] == "true":
            # empty only_true_has means no distinguishing signal
            if ("only_true_has = \n" in c["text_input"] or 
                "only_true_has = none" in c["text_input"].lower() or
                c["text_input"].count("only_true_has =") == 0):
                hard_queries.append(rec)

print(f"Queries with no unique true-tail relations: {len(hard_queries)}/331")

# Save as a harder eval set
with open("held_out_hard.json", "w") as f:
    json.dump(hard_queries, f, indent=2)

In [ ]:
evaluate("held_out_hard.json")

## with open("held_out_hard.json") as f:
    hard_data = json.load(f)

# Load per-query results to cross-reference
with open("inference_results.json") as f:
    inf = json.load(f)

# Build lookup from triple string to result
result_lookup = {r["triple"]: r for r in inf["per_query"]}

print("═══ HARD FAILURE CASES (hard_failure=True, no unique relations) ═══\n")
for rec in hard_data:
    if not rec["hard_failure"]:
        continue

    triple = rec["triple"]
    head, relation, true_tail = (
        [x.strip() for x in triple]
        if isinstance(triple, list)
        else [x.strip() for x in triple.split("|")]
    )
    key = f"{head}|{relation}|{true_tail}"
    res = result_lookup.get(key)

    # Find true tail candidate
    true_cand = next(c for c in rec["candidates"] if c["entity"] == true_tail)
    # Find what BERT ranked #1
    print(f"Query    : {head} | {relation} | ???")
    print(f"True tail: {true_tail}  (kge_rank={true_cand.get('kge_rank')}  kge_score={true_cand['features'].get('kge_score'):.3f})")
    if res:
        print(f"rank_bert={res['rank_bert']}  rank_kge={res['rank_kge']}  rank_comb={res['rank_comb']}")
        print(f"Top-3 combined: {res['top3_comb']}")
    print(f"Candidates: {[c['entity'] for c in rec['candidates']]}")
    print()